# 01 - Preparación del Dataset
**Objetivo:** Preparar y transformar el dataset que será usado para entrenar un modelo de IA aplicado a CMM. Este cuaderno prepara datos, define y guarda una función reutilizable `transform_data()` y guarda artefactos para los siguientes cuadernos.

**Nota:** No se entrena ningún modelo aquí.

## Requisitos y estructura de archivos
Se usan librerías estándar: `pandas`, `numpy`, `scikit-learn`, `joblib`.
Se crearán las carpetas `data/` y `models/` para almacenar artefactos.

In [1]:
# Imports y creación de carpetas
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import joblib

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Carpetas listas: data/ y models/')

Carpetas listas: data/ y models/


## Carga del dataset original
El cuaderno busca `data/original_dataset.csv`. Si no existe, simula un dataset de ejemplo reproducible.

In [2]:
# Cargar CIFAR-10 como dataset de ejemplo
try:
    from tensorflow.keras.datasets import cifar10
except Exception:
    raise ImportError('TensorFlow no está instalado en el entorno. Instala tensorflow y reintenta.')

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X = np.concatenate([X_train, X_test], axis=0)
y = np.concatenate([y_train, y_test], axis=0).reshape(-1)

print('CIFAR-10 cargado. Shapes -> X:', X.shape, 'y:', y.shape)

# Guardar como dataset procesado bruto para los siguientes cuadernos
np.savez_compressed('data/cifar10_raw.npz', X=X, y=y)
print('Guardado data/cifar10_raw.npz')

X.shape, y.shape

CIFAR-10 cargado. Shapes -> X: (60000, 32, 32, 3) y: (60000,)
Guardado data/cifar10_raw.npz


((60000, 32, 32, 3), (60000,))

## Análisis exploratorio básico
Veremos `shape`, tipos de datos y valores nulos.

In [4]:
# Análisis exploratorio básico para CIFAR-10
print('X shape:', X.shape)
print('X dtype:', X.dtype)
print('y shape:', y.shape)

# Distribución de etiquetas
labels, counts = np.unique(y, return_counts=True)
print('\nLabel distribution:')
for lab, cnt in zip(labels, counts):
    print(f'  label {int(lab)}: {int(cnt)} samples')

# Estadísticas simples sobre valores de píxel
print('\nPixel range: min=', X.min(), 'max=', X.max())
print('Mean pixel value (per channel):', X.mean(axis=(0,1,2)))


X shape: (60000, 32, 32, 3)
X dtype: uint8
y shape: (60000,)

Label distribution:
  label 0: 6000 samples
  label 1: 6000 samples
  label 2: 6000 samples
  label 3: 6000 samples
  label 4: 6000 samples
  label 5: 6000 samples
  label 6: 6000 samples
  label 7: 6000 samples
  label 8: 6000 samples
  label 9: 6000 samples

Pixel range: min= 0 max= 255
Mean pixel value (per channel): [125.42653861 123.07674582 114.03020576]


## Limpieza de datos
Estrategia:
- Columnas con más del 30% de valores faltantes se eliminan.
- Para numéricos: imputación por mediana.
- Para categóricos: imputación por moda.
- Se documenta cada decisión para reproducibilidad.

In [5]:
# Transformación de imágenes: normalización y aplanamiento
from utils.transform import transform_data_images

# Cargar los datos crudos guardados anteriormente
data_npz = np.load('data/cifar10_raw.npz')
X = data_npz['X']
y = data_npz['y']
print('Cargado CIFAR-10 bruto. Samples:', X.shape[0])

# Barajar para reproducibilidad
rng = np.random.default_rng(42)
idx = rng.permutation(X.shape[0])
X = X[idx]
y = y[idx]

# Ajustar transformador sobre el dataset y guardar artefactos
X_proc, y_proc, transform_info = transform_data_images(X, y, normalize=True, flatten=True, fit=True)
print('Transformación inicial realizada. Matriz X_proc shape:', X_proc.shape)

# Guardar dataset procesado completo y particiones
np.savez_compressed('data/processed_cifar10.npz', X=X_proc, y=y_proc)
from sklearn.model_selection import train_test_split
X_hist, X_new, y_hist, y_new = train_test_split(X_proc, y_proc, test_size=0.2, random_state=42, stratify=y_proc)
np.savez_compressed('data/historical.npz', X=X_hist, y=y_hist)
np.savez_compressed('data/new_samples.npz', X=X_new, y=y_new)
print('Datasets guardados en data/historical.npz y data/new_samples.npz')

Cargado CIFAR-10 bruto. Samples: 60000
Transformación inicial realizada. Matriz X_proc shape: (60000, 3072)
Datasets guardados en data/historical.npz y data/new_samples.npz


## Normalización / Escalado
Usaremos `StandardScaler` sobre columnas numéricas. La transformación se encapsula en `transform_data()` y se guarda.

In [8]:
# Ajustar transformador para imágenes si no existe
from utils.transform import transform_data_images, load_image_transformer
import numpy as np

if not os.path.exists('data/processed_cifar10.npz'):
    # Cargar raw y transformar
    if os.path.exists('data\\cifar10_raw.npz'):
        raw = np.load('data\cifar10_raw.npz')
        X_raw = raw['X']
        y_raw = raw['y']
        X_proc, y_proc, transform_info = transform_data_images(X_raw, y_raw, normalize=True, flatten=True, fit=True)
        np.savez_compressed('data/processed_cifar10.npz', X=X_proc, y=y_proc)
        print('Transformación inicial realizada. Matriz X_proc shape:', X_proc.shape)
    else:
        raise FileNotFoundError('No se encuentra data/cifar10_raw.npz. Ejecute la celda de carga de CIFAR-10.')
else:
    print('Processed dataset ya existe; cargar si se necesita con np.load("data/processed_cifar10.npz")')

Processed dataset ya existe; cargar si se necesita con np.load("data/processed_cifar10.npz")


## División en históricos y nuevos samples
Separa el dataset en `historical` (ej. 80%) y `new_samples` (20%) para simular nuevas observaciones que lleguen posteriormente.

In [10]:
# División en históricos y nuevos samples (para imágenes procesadas)
from sklearn.model_selection import train_test_split

if 'X_proc' in globals() and 'y_proc' in globals():
    X_all = X_proc
    y_all = y_proc
else:
    # Intentar cargar archivo procesado
    if os.path.exists('data/processed_cifar10.npz'):
        arr = np.load('data/processed_cifar10.npz')
        X_all = arr['X']
        y_all = arr['y']
    else:
        raise FileNotFoundError('No se encontró dataset procesado. Ejecute la celda de transformación o el cuaderno 01.')

X_hist, X_new, y_hist, y_new = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print('Historical shape:', X_hist.shape)
print('New samples shape:', X_new.shape)

# Guardar particiones para cuadernos posteriores
np.savez_compressed('data/historical.npz', X=X_hist, y=y_hist)
np.savez_compressed('data/new_samples.npz', X=X_new, y=y_new)
print('Datasets guardados en data/historical.npz y data/new_samples.npz')

Historical shape: (48000, 3072)
New samples shape: (12000, 3072)
Datasets guardados en data/historical.npz y data/new_samples.npz


## Notas finales y reproducibilidad
- `models/scaler.joblib` contiene el objeto `imputer` y `scaler` usado en `transform_data()`.
- Los cuadernos posteriores importarán `transform_data` desde `utils/transform.py` y cargarán los artefactos en `models/`.
- No se realizan entrenamientos aquí — solo preparación y guardado de artefactos.